 # Credit Risk Modeling: Data Preparation & Preprocessing
**Objective:** Initialize the reproducible modeling environment, establish reproducible seed governance (`seed=42`), and configure modular paths to the shared `src/` project modules.

In [ ]:
import sys
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, recall_score, precision_score

# Resolve Project Root
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "Notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    RAW_DATA_PATH,
    RANDOM_SEED,
    TEST_SIZE,
    V1_CORE_FEATURES,
    V1_BASELINE_METRICS,
)
from src.features import build_features, get_preprocessors

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", palette="muted")
print("Project root resolved:", PROJECT_ROOT.name)

## 1. Feature Engineering & Feature-Policy Choices
**Scope note:** The feature exclusions below are an illustrative modeling choice, not a legal compliance or fairness assessment.
- **Sensitive/proxy feature policy:** `CODE_GENDER`, `NAME_FAMILY_STATUS`, `NAME_TYPE_SUITE`, `CNT_CHILDREN`, and `CNT_FAM_MEMBERS` are excluded by default. Removing columns alone does not establish fairness.
- **Sparsity Mitigation:** 17 attributes with >60% missingness (primarily secondary housing metrics) are pruned.
- **Data Anomaly Resolution:** Legacy employment sentinel (`DAYS_EMPLOYED = 365,243`) is converted into an explicit binary flag (`EMPLOYED_ANOM`) to represent the sentinel separately.
- **Credit Ratios:** Financial leverage ratios (`LOAN_TO_INCOME`, `ANNUITY_TO_INCOME`, `CREDIT_TERM`) and a composite external-score feature (`EXT_SOURCE_MEAN`) are generated.

In [ ]:
print("Ingesting raw application dataset...")
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw shape: {df_raw.shape[0]:,} applications, {df_raw.shape[1]} attributes")

print("Executing build_features()...")
X, y = build_features(df_raw)
print(f"Transformed feature matrix shape: {X.shape}")
print(f"Default class prevalence: {y.mean():.4%} ({y.sum():,} total defaults)")

## 2. Frozen Baseline (v1) Reproduction Gate
**Validation Objective:** Train the benchmark 5-feature balanced Logistic Regression model on the stratified 80/20 training split and assert that holdout ROC-AUC matches the frozen Phase P0 baseline (0.7154) within a strict tolerance of $\vert{} \Delta \vert{} < 5 \times 10^{-4}$.

In [ ]:
# Exact 80/20 Stratified Partition matching v1 frozen split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
    stratify=y,
)

print(f"Training partition: {X_train.shape[0]:,} rows | Holdout partition: {X_test.shape[0]:,} rows")

# Fit Frozen v1 5-Feature Baseline Pipeline
v1_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(class_weight="balanced", random_state=RANDOM_SEED, max_iter=1000)),
])

print("Training baseline Logistic Regression on core 5 features...")
v1_pipeline.fit(X_train[V1_CORE_FEATURES], y_train)

# Evaluate Holdout Predictions
v1_probs = v1_pipeline.predict_proba(X_test[V1_CORE_FEATURES])[:, 1]
v1_preds = (v1_probs >= 0.5).astype(int)

v1_auc = roc_auc_score(y_test, v1_probs)
v1_recall = recall_score(y_test, v1_preds)
v1_precision = precision_score(y_test, v1_preds)
v1_pr_auc = average_precision_score(y_test, v1_probs)

print("\n==========================================")
print("     FROZEN V1 BASELINE RECONCILIATION    ")
print("==========================================")
print(f"ROC-AUC  : {v1_auc:.4f}  (Target: {V1_BASELINE_METRICS['roc_auc']:.4f})")
print(f"Recall   : {v1_recall:.2%}  (Target: {V1_BASELINE_METRICS['recall']:.2%})")
print(f"Precision: {v1_precision:.2%}  (Target: {V1_BASELINE_METRICS['precision']:.2%})")
print(f"PR-AUC   : {v1_pr_auc:.4f}  (Base Rate: {y_test.mean():.4f})")
print("==========================================")

# Mandatory Assertion Gate
tolerance = 5e-4
delta = abs(v1_auc - V1_BASELINE_METRICS["roc_auc"])
assert delta < tolerance, (
    f"Assertion Failed! Delta: {delta:.6f} >= {tolerance}. Verify train/test split or scaling."
)
print(f"\n[PASS] Sanity Check Succeeded: abs({v1_auc:.4f} - {V1_BASELINE_METRICS['roc_auc']:.4f}) = {delta:.6f} < 5e-4")

The previous notebook execution recorded this baseline check as passing. A clean rerun must reproduce it before the updated comparison is accepted.

## 3. Separate Preprocessing Pipelines
**Architectural Separation:**
- **`lr_preprocessor` (Linear Models):** Imputes missing values using median replacement with missingness indicators, standardizes numeric scales via $Z$-score, and applies low-frequency category pooling (`OneHotEncoder`).
- **`tree_preprocessor` (Tree Ensembles):** Preserves raw numerical values and `NaN`s for native tree split optimization; ordinally encodes categorical attributes.

In [ ]:
lr_preprocessor, tree_preprocessor = get_preprocessors(X)

print("Fitting lr_preprocessor on X_train...")
X_train_lr = lr_preprocessor.fit_transform(X_train)
X_test_lr = lr_preprocessor.transform(X_test)
print(f"-> Linear feature matrix shape: {X_train_lr.shape}")

print("Fitting tree_preprocessor on X_train...")
X_train_tree = tree_preprocessor.fit_transform(X_train)
X_test_tree = tree_preprocessor.transform(X_test)
print(f"-> Tree feature matrix shape:   {X_train_tree.shape}")

# Model Comparison: 5-Fold Stratified Cross-Validation
**Evaluation setup:**
- **Cross-Validation Setup:** 5-fold Stratified K-Fold executed on the 80% training set ($N = 246,008$) with fixed random seed (`42`).
- **Primary model-selection metric:** **PR-AUC (Average Precision)**, benchmarked against the empirical prevalence base rate of $0.0807$.
- **Validation Audit Metrics:** ROC-AUC, Gini Coefficient ($2 \times \text{AUC} - 1$), Kolmogorov-Smirnov (KS), Brier Score, and Log Loss.
- **Model Suite:**
  1. `LR_5feat`: Frozen 5-feature baseline benchmark (`class_weight='balanced'`).
  2. `LR_full`: Full feature regularized Logistic Regression ($L_2$ Ridge penalty, `class_weight='balanced'`).
  3. `RandomForest`: Bagged tree ensemble benchmark (`300 trees`, `max_depth=12`, `min_samples_leaf=50`).
  4. `LightGBM`: Gradient-boosted tree challenger (`scale_pos_weight=neg/pos`, `learning_rate=0.05`).
  5. `XGBoost`: Histogram-based gradient-boosted challenger (`scale_pos_weight=neg/pos`, `tree_method='hist'`).

In [ ]:
# Model Suite Imports and Imbalance Configuration
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

from src.evaluate import calculate_risk_metrics, cv_oof_evaluate
from src.config import RESULTS_DIR, ASSETS_DIR

# Calculate empirical class imbalance ratio for tree ensembles
neg_samples = (y_train == 0).sum()
pos_samples = (y_train == 1).sum()
scale_pos_weight = float(neg_samples / pos_samples)

print(f"Dataset Imbalance Configuration:")
print(f"• Non-defaults (Class 0): {neg_samples:,} | Defaults (Class 1): {pos_samples:,}")
print(f"• Target Imbalance Ratio (scale_pos_weight): {scale_pos_weight:.2f}")

cv_summary_records = []
oof_dict = {}

## 1. Linear Benchmarks: 5-Feature Baseline vs. Full Feature Logistic Regression
**Modeling rationale:**  
We evaluate the linear interpretability benchmark. The 5-feature baseline serves as our historical benchmark. The full-feature model applies an $L_2$ penalty ($C=0.1$) across all clean continuous variables, debt-burden ratios, and one-hot-encoded nominals to assess whether additional bureau signals yield meaningful margin separation.

In [ ]:
# 5-Fold CV Execution for Linear Models
# Model 1: 5-Feature Frozen Baseline LR
def make_lr_5feat_pipe():
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(class_weight="balanced", random_state=RANDOM_SEED, max_iter=1000)),
    ])

print("Executing 5-Fold CV: LR (5-Feature Baseline)...")
oof_lr5, scores_lr5, folds_lr5 = cv_oof_evaluate(
    model_name="lr_5feat",
    pipeline_factory=make_lr_5feat_pipe,
    X=X_train,
    y=y_train,
    feature_cols=V1_CORE_FEATURES,
)
cv_summary_records.append(scores_lr5)
oof_dict["lr_5feat"] = oof_lr5

print(f"-> LR 5-feat | CV ROC-AUC: {scores_lr5['roc_auc']:.4f} ± {scores_lr5['roc_auc_std']:.4f} | PR-AUC: {scores_lr5['pr_auc']:.4f} ± {scores_lr5['pr_auc_std']:.4f}")

# Model 2: Full Regularized Logistic Regression
def make_lr_full_pipe():
    return Pipeline([
        ("preprocessor", clone(lr_preprocessor)),
        ("classifier", LogisticRegression(
            C=0.1,
            penalty="l2",
            class_weight="balanced",
            solver="lbfgs",
            random_state=RANDOM_SEED,
            max_iter=1000,
        )),
    ])

print("\nExecuting 5-Fold CV: LR (Full Feature Space, L2 Regularized)...")
oof_lr_full, scores_lr_full, folds_lr_full = cv_oof_evaluate(
    model_name="lr_full",
    pipeline_factory=make_lr_full_pipe,
    X=X_train,
    y=y_train,
)
cv_summary_records.append(scores_lr_full)
oof_dict["lr_full"] = oof_lr_full

print(f"-> LR Full   | CV ROC-AUC: {scores_lr_full['roc_auc']:.4f} ± {scores_lr_full['roc_auc_std']:.4f} | PR-AUC: {scores_lr_full['pr_auc']:.4f} ± {scores_lr_full['pr_auc_std']:.4f}")

## 2. Bagged Tree Benchmark: Random Forest Classifier
**Architecture & Complexity Controls:**  
Random Forest captures non-linear interactions across debt-burden ratios without functional-form constraints. To avoid memorizing the majority class, we enforce `n_estimators=300`, `max_depth=12`, `min_samples_leaf=50`, and `class_weight='balanced_subsample'` (which recalibrates bootstrap weights dynamically per tree).

In [ ]:
# Cell 7: 5-Fold CV Execution for Random Forest
def make_rf_pipe():
    return Pipeline([
        ("preprocessor", clone(lr_preprocessor)),
        ("classifier", RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=50,
            max_features="sqrt",
            class_weight="balanced_subsample",
            random_state=RANDOM_SEED,
            n_jobs=-1,
        )),
    ])

print("Executing 5-Fold CV: Random Forest (300 Trees)...")
oof_rf, scores_rf, folds_rf = cv_oof_evaluate(
    model_name="random_forest",
    pipeline_factory=make_rf_pipe,
    X=X_train,
    y=y_train,
)
cv_summary_records.append(scores_rf)
oof_dict["random_forest"] = oof_rf

print(f"-> Random Forest | CV ROC-AUC: {scores_rf['roc_auc']:.4f} ± {scores_rf['roc_auc_std']:.4f} | PR-AUC: {scores_rf['pr_auc']:.4f} ± {scores_rf['pr_auc_std']:.4f}")

## 3. Gradient Boosted Challengers: LightGBM & XGBoost
**Non-Linear Optimization Strategy:**
Gradient boosted decision trees natively route missing value splits and handle asymmetric class risk via `scale_pos_weight = 11.39`:
- **LightGBM:** Leaf-wise expansion (`num_leaves=31`, `min_child_samples=100`, `learning_rate=0.05`, `subsample=0.8`, `colsample_bytree=0.7`).
- **XGBoost:** Depth-wise histogram splits (`tree_method='hist'`, `max_depth=5`, `min_child_weight=10`, `learning_rate=0.05`).

In [ ]:
# Cell 8: 5-Fold CV Execution for Gradient Boosted Trees
# Model 4: LightGBM Classifier
def make_lgb_pipe():
    return Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        ("classifier", lgb.LGBMClassifier(
            n_estimators=500,
            learning_rate=0.05,
            num_leaves=31,
            min_child_samples=100,
            subsample=0.8,
            colsample_bytree=0.7,
            scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED,
            n_jobs=-1,
            verbose=-1,
        )),
    ])

print("Executing 5-Fold CV: LightGBM Classifier...")
oof_lgb, scores_lgb, folds_lgb = cv_oof_evaluate(
    model_name="lightgbm",
    pipeline_factory=make_lgb_pipe,
    X=X_train,
    y=y_train,
)
cv_summary_records.append(scores_lgb)
oof_dict["lightgbm"] = oof_lgb

print(f"-> LightGBM  | CV ROC-AUC: {scores_lgb['roc_auc']:.4f} ± {scores_lgb['roc_auc_std']:.4f} | PR-AUC: {scores_lgb['pr_auc']:.4f} ± {scores_lgb['pr_auc_std']:.4f}")

# Model 5: XGBoost Classifier (Histogram-based)
def make_xgb_pipe():
    return Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        ("classifier", xgb.XGBClassifier(
            n_estimators=500,
            max_depth=5,
            learning_rate=0.05,
            min_child_weight=10,
            subsample=0.8,
            colsample_bytree=0.7,
            scale_pos_weight=scale_pos_weight,
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=-1,
            eval_metric="logloss",
        )),
    ])

print("\nExecuting 5-Fold CV: XGBoost Classifier...")
oof_xgb, scores_xgb, folds_xgb = cv_oof_evaluate(
    model_name="xgboost",
    pipeline_factory=make_xgb_pipe,
    X=X_train,
    y=y_train,
)
cv_summary_records.append(scores_xgb)
oof_dict["xgboost"] = oof_xgb

print(f"-> XGBoost   | CV ROC-AUC: {scores_xgb['roc_auc']:.4f} ± {scores_xgb['roc_auc_std']:.4f} | PR-AUC: {scores_xgb['pr_auc']:.4f} ± {scores_xgb['pr_auc_std']:.4f}")

## 4. Multi-Model Scorecard Compilation & Comparative Curves
**Audit Artifact Generation:**  
Consolidate all out-of-fold metrics into the official comparative scorecard, archive `results/cv_metrics.csv`, and render the joint 5-fold OOF Precision-Recall and ROC curve overlay (`assets/pr_roc_overlay.png`).

In [ ]:
# Cell 9: Scorecard Table Export and Discrimination Overlays
from sklearn.metrics import precision_recall_curve, roc_curve

df_cv_results = pd.DataFrame(cv_summary_records)
df_cv_results = df_cv_results.sort_values(by="pr_auc", ascending=False).reset_index(drop=True)

# Save official cross-validation scorecard
cv_metrics_csv_path = RESULTS_DIR / "cv_metrics.csv"
df_cv_results.to_csv(cv_metrics_csv_path, index=False)
fold_frames = []
for name, fold_df in [
    ("lr_5feat", folds_lr5), ("lr_full", folds_lr_full), ("random_forest", folds_rf),
    ("lightgbm", folds_lgb), ("xgboost", folds_xgb),
]:
    tmp = fold_df.copy()
    tmp["model"] = name
    fold_frames.append(tmp)
pd.concat(fold_frames, ignore_index=True).to_csv(RESULTS_DIR / "cv_fold_metrics.csv", index=False)

print("\n=========================================================================================")
print("                   OFFICIAL 5-FOLD CROSS-VALIDATION MODEL SCORECARD                      ")
print("=========================================================================================")
display_cols = ["model", "pr_auc", "pr_auc_std", "roc_auc", "roc_auc_std", "gini", "ks_stat", "brier_score", "train_time_sec"]
print(df_cv_results[display_cols].to_string(index=False))
print("=========================================================================================")
print("Scorecard successfully exported to results/cv_metrics.csv")

# Plotting: Joint Precision-Recall and ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 6), dpi=120)

colors = {
    "lightgbm": "#1f77b4",
    "xgboost": "#ff7f0e",
    "random_forest": "#2ca02c",
    "lr_full": "#d62728",
    "lr_5feat": "#9467bd",
}

# Subplot 1: Precision-Recall Curves
for model_name, oof_probs in oof_dict.items():
    prec, rec, _ = precision_recall_curve(y_train, oof_probs)
    pr_score = average_precision_score(y_train, oof_probs)
    axes[0].plot(rec, prec, label=f"{model_name} (PR-AUC = {pr_score:.4f})", color=colors.get(model_name, "gray"), lw=2)

axes[0].axhline(y=y_train.mean(), color="black", linestyle="--", label=f"Random Baseline ({y_train.mean():.4f})")
axes[0].set_title("5-Fold OOF Precision-Recall Curves", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Recall (True Positive Rate)", fontsize=11)
axes[0].set_ylabel("Precision (Positive Predictive Value)", fontsize=11)
axes[0].set_xlim([0.0, 1.0])
axes[0].set_ylim([0.0, 1.0])
axes[0].legend(loc="upper right", frameon=True)

# Subplot 2: ROC Curves
for model_name, oof_probs in oof_dict.items():
    fpr, tpr, _ = roc_curve(y_train, oof_probs)
    roc_score = roc_auc_score(y_train, oof_probs)
    axes[1].plot(fpr, tpr, label=f"{model_name} (ROC-AUC = {roc_score:.4f})", color=colors.get(model_name, "gray"), lw=2)

axes[1].plot([0, 1], [0, 1], color="black", linestyle="--", label="Random Chance (0.5000)")
axes[1].set_title("5-Fold OOF ROC Discrimination Curves", fontsize=12, fontweight="bold")
axes[1].set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
axes[1].set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.0])
axes[1].legend(loc="lower right", frameon=True)

plt.tight_layout()
overlay_plot_path = ASSETS_DIR / "pr_roc_overlay.png"
plt.savefig(overlay_plot_path, dpi=300)
plt.show()
print("Comparative curve overlay exported to assets/pr_roc_overlay.png")

## 5. Champion Selection
**Selection rule:**  
The candidate with the highest **pooled out-of-fold PR-AUC**, computed across all training rows predicted out of fold is formally designated as the portfolio **Champion Model**. The selection rule identifies the highest OOF PR-AUC; statistical uncertainty is evaluated separately.

In [ ]:
# Cell 10: Champion identification
champion_record = df_cv_results.iloc[0]
champion_name = champion_record["model"]
champion_pr_auc = champion_record["pr_auc"]
champion_roc_auc = champion_record["roc_auc"]

baseline_record = df_cv_results[df_cv_results["model"] == "lr_5feat"].iloc[0]
baseline_pr_auc = baseline_record["pr_auc"]
baseline_roc_auc = baseline_record["roc_auc"]

pr_auc_lift = ((champion_pr_auc - baseline_pr_auc) / baseline_pr_auc) * 100
roc_auc_lift = ((champion_roc_auc - baseline_roc_auc) / baseline_roc_auc) * 100

print("=========================================================================================")
print(f"CHAMPION SELECTED BY POOLED OOF PR-AUC: {champion_name.upper()}")
print("=========================================================================================")
print(f"• Champion PR-AUC: {champion_pr_auc:.4f} vs. Baseline: {baseline_pr_auc:.4f} (+{pr_auc_lift:.2f}% Relative Lift)")
print(f"• Champion ROC-AUC: {champion_roc_auc:.4f} vs. Baseline: {baseline_roc_auc:.4f} (+{roc_auc_lift:.2f}% Relative Lift)")
print(f"• Gini Coefficient: {champion_record['gini']:.4f}")
print(f"• KS Separation:   {champion_record['ks_stat']:.4f}")
print("=========================================================================================")

# Governance Gate: Champion must strictly beat the baseline on primary metric
assert champion_pr_auc > baseline_pr_auc, "FATAL: Champion candidate failed to beat the baseline benchmark!"
print("Champion selected from the training-side OOF ranking; statistical uncertainty is assessed separately.")

# Probability Calibration, Illustrative Decision Costs & Bootstrap Inference
## 1. Out-of-Fold Isotonic Probability Calibration
**Calibration and illustrative decision-cost methodology:**  
Training tree ensembles with `scale_pos_weight = 11.39` shifts the output probability distribution upward to artificially penalize defaults. Applying a threshold directly to uncalibrated weighted scores can produce poorly motivated decisions. We fit an **Isotonic Regression** model on the Champion's out-of-fold predictions ($N = 246,008$) and verify calibration via Brier score reduction and reliability diagrams.

In [ ]:
# Cell 11: Isotonic Calibration & Reliability Curve Generation
from sklearn.calibration import calibration_curve
from src.costs import calibrate_probabilities_isotonic, optimize_threshold_grid

# Retrain Champion model on full training set to score holdout test set
print(f"Fitting selected Champion pipeline ({champion_name}) on X_train...")
model_factories = {"xgboost": make_xgb_pipe, "lightgbm": make_lgb_pipe}
assert champion_name in model_factories, f"Unsupported champion: {champion_name}"
runner_up_name = "lightgbm" if champion_name == "xgboost" else "xgboost"
champ_pipeline = model_factories[champion_name]()
champ_pipeline.fit(X_train, y_train)

# Raw test set predictions
test_raw_probs = champ_pipeline.predict_proba(X_test)[:, 1]

# Calibrate using Champion OOF predictions
oof_raw_probs = oof_dict[champion_name]
iso_calibrator, oof_calib_probs, test_calib_probs, brier_audit = calibrate_probabilities_isotonic(
    y_train_true=y_train.to_numpy(),
    oof_train_probs=oof_raw_probs,
    test_raw_probs=test_raw_probs,
)

# Fit and calibrate the runner-up separately so comparisons never mix model scores.
runner_up_pipeline = model_factories[runner_up_name]()
runner_up_pipeline.fit(X_train, y_train)
runner_up_raw_probs = runner_up_pipeline.predict_proba(X_test)[:, 1]
_, runner_up_oof_calib_probs, runner_up_test_calib_probs, runner_up_brier_audit = calibrate_probabilities_isotonic(
    y_train_true=y_train.to_numpy(),
    oof_train_probs=oof_dict[runner_up_name],
    test_raw_probs=runner_up_raw_probs,
)
runner_up_t_star, _, runner_up_threshold_table = optimize_threshold_grid(
    y_true=y_train.to_numpy(),
    calib_probs=runner_up_oof_calib_probs,
    amt_credit=X_train["AMT_CREDIT"].to_numpy(),
    r_ratio=5.0,
)

print("\n========================================================")
print("           ISOTONIC PROBABILITY CALIBRATION AUDIT       ")
print("========================================================")
print(f"• Brier Score (Raw OOF)       : {brier_audit['brier_before']:.4f}")
print(f"• Brier Score (Calibrated OOF): {brier_audit['brier_after']:.4f}")
print(f"• Calibration Error Reduction : {brier_audit['brier_improvement_pct']:.2f}%")
print("========================================================")

# Render Reliability Diagram (Calibration Curve)
prob_true_raw, prob_pred_raw = calibration_curve(y_train, oof_raw_probs, n_bins=10)
prob_true_calib, prob_pred_calib = calibration_curve(y_train, oof_calib_probs, n_bins=10)

plt.figure(figsize=(7, 6), dpi=120)
plt.plot([0, 1], [0, 1], "k--", label="Perfect Calibration")
plt.plot(prob_pred_raw, prob_true_raw, "s-", color="#d62728", label=f"Raw OOF (Brier = {brier_audit['brier_before']:.4f})")
plt.plot(prob_pred_calib, prob_true_calib, "o-", color="#1f77b4", lw=2, label=f"Isotonic Calibrated (Brier = {brier_audit['brier_after']:.4f})")
plt.title("Reliability Diagram: Champion Probability Calibration", fontsize=12, fontweight="bold")
plt.xlabel("Mean Predicted Probability", fontsize=11)
plt.ylabel("Fraction of True Defaults (Empirical)", fontsize=11)
plt.legend(loc="upper left")
plt.tight_layout()

calib_curve_path = ASSETS_DIR / "calibration_curve.png"
plt.savefig(calib_curve_path, dpi=300)
plt.show()
print("Calibration reliability diagram archived to assets/calibration_curve.png")

## 2. Illustrative Decision-Cost Function & Threshold Search
**Illustrative decision-cost assumptions:**  
For this illustrative decision-cost exercise, we assume $\text{LGD} = 0.45$ and an opportunity cost of rejection of $0.09$, establishing an asymmetric cost ratio:
$$r = \frac{\text{LGD}}{\text{Margin}} = \frac{0.45}{0.09} = 5.0$$
Under these simplified assumptions, the analytic probability cutoff is:
$$t^* = \frac{1}{1 + r} = \frac{1}{1 + 5.0} \approx 0.1667$$
We execute an empirical grid search over calibrated OOF probabilities ($0.01 \le t \le 0.99$, step $0.005$) and verify consistency against the analytic benchmark. We then construct the cost-sensitivity matrix across $r \in \{2, 3, 5, 10, 20\}$ and archive `results/threshold_table.csv`.

In [ ]:
# Cell 12: Threshold Grid Optimization & Sensitivity Analysis
from src.costs import optimize_threshold_grid, generate_sensitivity_table, compute_credit_cost

amt_credit_train = X_train["AMT_CREDIT"].to_numpy()

# 1. Empirical Grid Search for r = 5.0
emp_t_star, ana_t_star, df_grid_search = optimize_threshold_grid(
    y_true=y_train.to_numpy(),
    calib_probs=oof_calib_probs,
    amt_credit=amt_credit_train,
    r_ratio=5.0,
)

print("========================================================")
print("         THRESHOLD OPTIMIZATION RESULTS (r = 5.0)       ")
print("========================================================")
print(f"• Empirical Minimum Threshold (t*) : {emp_t_star:.4f}")
print(f"• Analytic Bayes Threshold (1/(1+r)): {ana_t_star:.4f}")
print(f"• Absolute Discrepancy (|t* - ana|): {abs(emp_t_star - ana_t_star):.4f}")
print("========================================================")

# 2. Sensitivity Analysis Table across r in {2, 3, 5, 10, 20}
df_sensitivity = generate_sensitivity_table(
    y_true=y_train.to_numpy(),
    calib_probs=oof_calib_probs,
    amt_credit=amt_credit_train,
)

thresh_table_path = RESULTS_DIR / "threshold_table.csv"
df_sensitivity.to_csv(thresh_table_path, index=False)

print("\n--- Cost Sensitivity Table (r in {2, 3, 5, 10, 20}) ---")
print(df_sensitivity.to_string(index=False))
print(f"\nSensitivity table saved to: {thresh_table_path}")

In [ ]:
# Select a baseline policy threshold using only cross-fitted training predictions.
from src.costs import calibrate_probabilities_isotonic, optimize_threshold_grid
_, oof_v1_calib_probs, v1_test_calib_probs, v1_calibration_audit = calibrate_probabilities_isotonic(
    y_train_true=y_train.to_numpy(),
    oof_train_probs=oof_dict["lr_5feat"],
    test_raw_probs=v1_probs,
)
v1_t_star, _, _ = optimize_threshold_grid(
    y_true=y_train.to_numpy(),
    calib_probs=oof_v1_calib_probs,
    amt_credit=X_train["AMT_CREDIT"].to_numpy(),
    r_ratio=5.0,
)
print(f"Baseline v1 own OOF-calibrated cost-optimal threshold: {v1_t_star:.3f}")

## 3. Holdout Test Evaluation with a Locked Policy
**Validation protocol:** The decision thresholds are chosen using training-side out-of-fold predictions. The holdout partition is used only for final reporting. Five policies are compared:
1. **Approve everyone:** all test applicants are approved (equivalent to threshold 1.0 because calibrated probabilities are capped below 1).
2. **Baseline v1:** frozen five-feature logistic regression at threshold 0.50.
3. **Baseline v1, own threshold:** baseline calibrated on OOF predictions with threshold selected on training data.
4. **Selected tree model, raw-score reference:** XGBoost or LightGBM at raw score threshold 0.50; this is not a calibrated policy.
5. **Selected tree model, calibrated policy:** the selected champion at the threshold chosen from cross-fitted training predictions.

In [ ]:
# Cell 13: Holdout Test Set Scoring & Point Evaluation
amt_credit_test = X_test["AMT_CREDIT"].to_numpy()
y_test_vec = y_test.to_numpy()

# Test evaluation for Baseline v1 (using v1_probs from Phase P1)
v1_perf_test = compute_credit_cost(y_test_vec, v1_probs, threshold=0.5, amt_credit=amt_credit_test)

# The raw-score 0.50 row is retained only as a reference because weighted scores are not calibrated probabilities.
champ_std_perf = compute_credit_cost(y_test_vec, test_raw_probs, threshold=0.5, amt_credit=amt_credit_test)
champ_opt_perf = compute_credit_cost(y_test_vec, test_calib_probs, threshold=emp_t_star, amt_credit=amt_credit_test)

# Approve Everyone Baseline (Threshold = 1.0)
approve_all_perf = compute_credit_cost(y_test_vec, test_calib_probs, threshold=1.0, amt_credit=amt_credit_test)

# Export Scored Test Data
df_test_scored = pd.DataFrame({
    "y_true": y_test_vec,
    "amt_credit": amt_credit_test,
    "v1_prob": v1_probs,
    "champ_raw_prob": test_raw_probs,
    "champ_calib_prob": test_calib_probs,
    "champ_pred_optimal": (test_calib_probs >= emp_t_star).astype(int),
    "runner_up_raw_prob": runner_up_raw_probs,
    "runner_up_calib_prob": runner_up_test_calib_probs,
    "runner_up_pred_optimal": (runner_up_test_calib_probs >= runner_up_t_star).astype(int),
})

test_scored_path = RESULTS_DIR / "test_scored.csv"
df_test_scored.to_csv(test_scored_path, index=False)
print("Scored holdout test predictions archived to: results/test_scored.csv")

In [ ]:
# Test calibration diagnostics on the locked holdout set (reporting only; no tuning here).
from sklearn.metrics import brier_score_loss
observed_prevalence_test = float(np.mean(y_test_vec))
test_calibration_rows = []
for label, score in [
    (f"{champion_name}: raw model score (not calibrated)", test_raw_probs),
    (f"{champion_name}: isotonic-calibrated probability", test_calib_probs),
    (f"{runner_up_name}: raw model score (not calibrated)", runner_up_raw_probs),
    (f"{runner_up_name}: isotonic-calibrated probability", runner_up_test_calib_probs),
]:
    test_calibration_rows.append({
        "score_type": label,
        "brier_score": float(brier_score_loss(y_test_vec, score)),
        "mean_predicted_probability": float(np.mean(score)),
        "observed_prevalence": observed_prevalence_test,
        "n_unique_scores": int(pd.Series(score).nunique()),
        "n_exact_zeros": int(np.sum(score == 0.0)),
        "n_exact_ones": int(np.sum(score == 1.0)),
    })
df_test_calibration = pd.DataFrame(test_calibration_rows)
df_test_calibration.to_csv(RESULTS_DIR / "test_calibration.csv", index=False)
print("Test calibration diagnostics saved to results/test_calibration.csv")
print(df_test_calibration.to_string(index=False))

## 4. 1,000 Paired Bootstrap Resamples on Test Set (95% Confidence Intervals)
**Statistical Significance Assessment:**  
Cross-validation fold $t$-tests underestimate standard error because folds share training data. We execute 1,000 paired bootstrap iterations on the holdout test set to compute non-parametric 95% Confidence Intervals for ROC-AUC, PR-AUC, Expected Cost per applicant, and paired differences against approve-all, v1 at 0.50, and v1 with its own training-selected threshold.

In [ ]:
# Cell 14: Bootstrap Resampling and Paired Inference
from src.evaluate import bootstrap_test_evaluation

# Model Dictionary for Bootstrap Harness
bootstrap_model_dict = {
    "Approve everyone": (test_calib_probs, 1.0),
    "Baseline v1 (5-feature LR)": (v1_probs, 0.5),
    "Baseline v1 (OOF-calibrated cost-optimal t*)": (v1_test_calib_probs, v1_t_star),
    "Champion (t=0.5)": (test_raw_probs, 0.5),
    "Champion (cost-optimal t*)": (test_calib_probs, emp_t_star),
    "Runner-up (cost-optimal t*)": (runner_up_test_calib_probs, runner_up_t_star),
}

print("Running 1,000 paired bootstrap resamples on test set (this takes ~1-2 minutes)...")
df_ci_summary, df_diff_summary = bootstrap_test_evaluation(
    y_test=y_test_vec,
    amt_credit_test=amt_credit_test,
    model_predictions=bootstrap_model_dict,
    n_bootstraps=1000,
    random_seed=RANDOM_SEED,
)

test_metrics_path = RESULTS_DIR / "test_metrics.csv"
df_ci_summary.to_csv(test_metrics_path, index=False)
df_diff_summary.to_csv(RESULTS_DIR / "test_paired_diffs.csv", index=False)

print("\n=========================================================================================")
print("             1,000 BOOTSTRAP RESAMPLE EVALUATION SUMMARY (95% CONFIDENCE INTERVALS)     ")
print("=========================================================================================")
print(df_ci_summary.to_string(index=False))
print("=========================================================================================")
print(f"Test metrics archived to: {test_metrics_path}")

## 5. Expected Decision-Cost Comparison
**Interpretation:**  
We construct a comparison table and compute the headline metric: **expected decision cost under the stated assumptions, with comparisons against approve-all and two baseline policies**.

In [ ]:
# Cell 15: Point estimates and paired test-set comparisons
v1_optimal_perf = compute_credit_cost(
    y_test_vec, v1_test_calib_probs, threshold=v1_t_star, amt_credit=amt_credit_test
)

def make_headline_record(policy, decision_threshold, performance):
    return {
        "policy": policy,
        "decision_threshold": decision_threshold,
        **{key: value for key, value in performance.items() if key != "threshold"},
    }

headline_costs = pd.DataFrame([
    make_headline_record("Approve everyone", 1.0, approve_all_perf),
    make_headline_record("Baseline v1 (threshold 0.50)", 0.50, v1_perf_test),
    make_headline_record("Baseline v1 (OOF-calibrated cost-optimal)", v1_t_star, v1_optimal_perf),
    make_headline_record(f"Champion {champion_name} (raw weighted score; reference only)", 0.50, champ_std_perf),
    make_headline_record(f"Champion {champion_name} (OOF-calibrated cost-optimal)", emp_t_star, champ_opt_perf),
    make_headline_record(f"Runner-up {runner_up_name} (OOF-calibrated cost-optimal)", runner_up_t_star,
                         compute_credit_cost(y_test_vec, runner_up_test_calib_probs, threshold=runner_up_t_star, amt_credit=amt_credit_test)),
])
headline_costs.to_csv(RESULTS_DIR / "headline_costs.csv", index=False)
operating_cols = [
    "policy", "decision_threshold", "approval_rate", "default_rate_approved",
    "recall", "precision", "cost_per_applicant", "cost_pct_vs_approve_all",
]
headline_costs[operating_cols].to_csv(RESULTS_DIR / "test_operating_points.csv", index=False)

approve_all_cost = float(approve_all_perf["cost_per_applicant"])
v1_default_cost = float(v1_perf_test["cost_per_applicant"])
v1_optimal_cost = float(v1_optimal_perf["cost_per_applicant"])
champion_cost = float(champ_opt_perf["cost_per_applicant"])

print("=== Expected decision cost (illustrative assumptions; dataset currency units) ===")
print(headline_costs[operating_cols].to_string(index=False))
print(f"Champion selected from training OOF PR-AUC: {champion_name}")
print(f"Savings vs approve-all: {approve_all_cost-champion_cost:,.2f} currency units/application ({(approve_all_cost-champion_cost)/approve_all_cost*100:.2f}%)")
print(f"Savings vs v1 at threshold 0.50: {v1_default_cost-champion_cost:,.2f} currency units/application ({(v1_default_cost-champion_cost)/v1_default_cost*100:.2f}%)")
print(f"Savings vs v1 own cost-optimal threshold: {v1_optimal_cost-champion_cost:,.2f} currency units/application ({(v1_optimal_cost-champion_cost)/v1_optimal_cost*100:.2f}%)")
print("Paired bootstrap comparison intervals saved to results/test_paired_diffs.csv")


# Class-Imbalance Strategy Comparison
**Question:**  
We empirically test whether synthetic oversampling (SMOTE) or algorithmic loss reweighting (`scale_pos_weight` / `class_weight`) alters ranking discrimination once probabilities are calibrated and cost-optimized.
- **Hypothesis:** Resampling alters the intercept and calibration rather than rank ordering. After Isotonic calibration and cost-optimal threshold tuning ($t^*$), the expected decision costs across strategies will be similar.
- **Configurations tested (5 runs):**
  1. `LR (full) - Unweighted`: Logistic Regression (`class_weight=None`).
  2. `XGBoost - Unweighted`: XGBoost with `scale_pos_weight=1.0`.
  3. `LR (full) - Weighted`: Baseline balanced weighting (`class_weight='balanced'`).
  4. `XGBoost - Loss-Weighted`: XGBoost with `scale_pos_weight=11.39` and its own OOF calibration/threshold.
  5. `LR (full) - SMOTE`: 50% synthetic minority oversampling (`sampling_strategy=0.5`) applied exclusively to training folds inside an `imblearn.pipeline.Pipeline` to prevent synthetic leakage.
  *Scope note: SMOTE is tested only with Logistic Regression. It can create fractional one-hot values, so treat this run as exploratory; a tree-plus-SMOTE comparison was not run.**

In [ ]:
# Cell 16: Class-imbalance strategy comparison
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from src.costs import calibrate_probabilities_isotonic, optimize_threshold_grid, compute_credit_cost

imbalance_results = []
amt_credit_train = X_train["AMT_CREDIT"].to_numpy()
y_train_vec = y_train.to_numpy()

# -----------------------------------------------------------------------------
# Run 1: LR (Full) - Unweighted (class_weight=None)
# -----------------------------------------------------------------------------
print("Executing Run 1/5: LR Full - Unweighted...")
def make_lr_unweighted():
    return Pipeline([
        ("preprocessor", clone(lr_preprocessor)),
        ("classifier", LogisticRegression(C=0.1, penalty="l2", class_weight=None, solver="lbfgs", random_state=RANDOM_SEED, max_iter=1000)),
    ])
oof_lr_unw, scores_lr_unw, _ = cv_oof_evaluate("lr_unweighted", make_lr_unweighted, X_train, y_train, save_oof=False)
_, oof_calib_lr_unw, _, _ = calibrate_probabilities_isotonic(y_train_vec, oof_lr_unw)
t_lr_unw, _, _ = optimize_threshold_grid(y_train_vec, oof_calib_lr_unw, amt_credit_train, r_ratio=5.0)
cost_lr_unw = compute_credit_cost(y_train_vec, oof_calib_lr_unw, threshold=t_lr_unw, amt_credit=amt_credit_train)

imbalance_results.append({
    "strategy": "None (Unweighted)",
    "model": "LR (full)",
    "roc_auc": scores_lr_unw["roc_auc"],
    "pr_auc": scores_lr_unw["pr_auc"],
    "brier_raw": scores_lr_unw["brier_score"],
    "optimal_threshold": t_lr_unw,
    "cost_per_applicant": cost_lr_unw["cost_per_applicant"],
    "cost_pct_vs_approve_all": cost_lr_unw["cost_pct_vs_approve_all"],
})

# -----------------------------------------------------------------------------
# Run 2: Champion (XGBoost) - Unweighted (scale_pos_weight=1.0)
# -----------------------------------------------------------------------------
print("Executing Run 2/5: Champion XGBoost - Unweighted...")
def make_xgb_unweighted():
    return Pipeline([
        ("preprocessor", clone(tree_preprocessor)),
        ("classifier", xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05, min_child_weight=10,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=1.0,
            tree_method="hist", random_state=RANDOM_SEED, n_jobs=-1, eval_metric="logloss"
        )),
    ])
oof_xgb_unw, scores_xgb_unw, _ = cv_oof_evaluate("xgb_unweighted", make_xgb_unweighted, X_train, y_train, save_oof=False)
_, oof_calib_xgb_unw, _, _ = calibrate_probabilities_isotonic(y_train_vec, oof_xgb_unw)
t_xgb_unw, _, _ = optimize_threshold_grid(y_train_vec, oof_calib_xgb_unw, amt_credit_train, r_ratio=5.0)
cost_xgb_unw = compute_credit_cost(y_train_vec, oof_calib_xgb_unw, threshold=t_xgb_unw, amt_credit=amt_credit_train)

imbalance_results.append({
    "strategy": "None (Unweighted)",
    "model": "XGBoost (unweighted)",
    "roc_auc": scores_xgb_unw["roc_auc"],
    "pr_auc": scores_xgb_unw["pr_auc"],
    "brier_raw": scores_xgb_unw["brier_score"],
    "optimal_threshold": t_xgb_unw,
    "cost_per_applicant": cost_xgb_unw["cost_per_applicant"],
    "cost_pct_vs_approve_all": cost_xgb_unw["cost_pct_vs_approve_all"],
})

# -----------------------------------------------------------------------------
# Run 3: LR (Full) - Class Weights ('balanced') [Reusing P2/P3 Artifacts]
# -----------------------------------------------------------------------------
print("Executing Run 3/5: LR Full - Class Weights (Balanced)...")
oof_calib_lr_w = calibrate_probabilities_isotonic(y_train_vec, oof_dict["lr_full"])[1]
t_lr_w, _, _ = optimize_threshold_grid(y_train_vec, oof_calib_lr_w, amt_credit_train, r_ratio=5.0)
cost_lr_w = compute_credit_cost(y_train_vec, oof_calib_lr_w, threshold=t_lr_w, amt_credit=amt_credit_train)
score_lr_full = df_cv_results[df_cv_results["model"] == "lr_full"].iloc[0]

imbalance_results.append({
    "strategy": "Class Weights (Balanced)",
    "model": "LR (full)",
    "roc_auc": score_lr_full["roc_auc"],
    "pr_auc": score_lr_full["pr_auc"],
    "brier_raw": score_lr_full["brier_score"],
    "optimal_threshold": t_lr_w,
    "cost_per_applicant": cost_lr_w["cost_per_applicant"],
    "cost_pct_vs_approve_all": cost_lr_w["cost_pct_vs_approve_all"],
})

# -----------------------------------------------------------------------------
# Run 4: XGBoost - Scale-Pos-Weight (11.39); use XGBoost's own OOF scores and threshold.
# -----------------------------------------------------------------------------
print("Executing Run 4/5: XGBoost - Loss Weighted (11.39)...")
_, oof_calib_xgb_weighted, _, _ = calibrate_probabilities_isotonic(
    y_train_vec, oof_dict["xgboost"]
)
t_xgb_weighted, _, _ = optimize_threshold_grid(
    y_train_vec, oof_calib_xgb_weighted, amt_credit_train, r_ratio=5.0
)
cost_xgb_weighted = compute_credit_cost(
    y_train_vec, oof_calib_xgb_weighted, threshold=t_xgb_weighted, amt_credit=amt_credit_train
)
score_xgb = df_cv_results[df_cv_results["model"] == "xgboost"].iloc[0]

imbalance_results.append({
    "strategy": "Loss Weights (scale_pos_weight)",
    "model": "XGBoost (loss-weighted)",
    "roc_auc": score_xgb["roc_auc"],
    "pr_auc": score_xgb["pr_auc"],
    "brier_raw": score_xgb["brier_score"],
    "optimal_threshold": t_xgb_weighted,
    "cost_per_applicant": cost_xgb_weighted["cost_per_applicant"],
    "cost_pct_vs_approve_all": cost_xgb_weighted["cost_pct_vs_approve_all"],
})

# -----------------------------------------------------------------------------
# Run 5: LR (Full) - SMOTE (sampling_strategy=0.5 on training folds)
# -----------------------------------------------------------------------------
print("Executing Run 5/5: LR Full - SMOTE (sampling_strategy=0.5)...")
def make_lr_smote():
    return ImbPipeline([
        ("preprocessor", clone(lr_preprocessor)),
        ("smote", SMOTE(sampling_strategy=0.5, random_state=RANDOM_SEED)),
        ("classifier", LogisticRegression(C=0.1, penalty="l2", class_weight=None, solver="lbfgs", random_state=RANDOM_SEED, max_iter=1000)),
    ])
oof_lr_smote, scores_lr_smote, _ = cv_oof_evaluate("lr_smote", make_lr_smote, X_train, y_train, save_oof=False)
_, oof_calib_lr_smote, _, _ = calibrate_probabilities_isotonic(y_train_vec, oof_lr_smote)
t_lr_smote, _, _ = optimize_threshold_grid(y_train_vec, oof_calib_lr_smote, amt_credit_train, r_ratio=5.0)
cost_lr_smote = compute_credit_cost(y_train_vec, oof_calib_lr_smote, threshold=t_lr_smote, amt_credit=amt_credit_train)

imbalance_results.append({
    "strategy": "SMOTE (ratio=0.5)",
    "model": "LR (full)",
    "roc_auc": scores_lr_smote["roc_auc"],
    "pr_auc": scores_lr_smote["pr_auc"],
    "brier_raw": scores_lr_smote["brier_score"],
    "optimal_threshold": t_lr_smote,
    "cost_per_applicant": cost_lr_smote["cost_per_applicant"],
    "cost_pct_vs_approve_all": cost_lr_smote["cost_pct_vs_approve_all"],
})

print("All 5 imbalance benchmark runs completed.")

## Imbalance Study Summary
**Output:**  
We compile the official 5-row comparative matrix, save `results/imbalance_study.csv`, and evaluate the hypothesis that sampling strategies converge once probabilities are calibrated.

In [ ]:
# Cell 17: Imbalance scorecard compilation
df_imbalance_study = pd.DataFrame(imbalance_results)

# Order table: None -> Weights -> SMOTE
df_imbalance_study["sort_key"] = [1, 2, 3, 4, 5]
df_imbalance_study = df_imbalance_study.sort_values("sort_key").drop(columns=["sort_key"]).reset_index(drop=True)

imbalance_csv_path = RESULTS_DIR / "imbalance_study.csv"
df_imbalance_study.to_csv(imbalance_csv_path, index=False)

print("\n=========================================================================================")
print("             CLASS-IMBALANCE BENCHMARK MATRIX (5 RUNS)                 ")
print("=========================================================================================")
display_cols = ["strategy", "model", "pr_auc", "roc_auc", "brier_raw", "optimal_threshold", "cost_per_applicant", "cost_pct_vs_approve_all"]
print(df_imbalance_study[display_cols].to_string(index=False))
print("=========================================================================================")
print(f"Results archived to: {imbalance_csv_path}")

# Phase P4 Institutional Conclusions (current project Requirement)
print("\n--- Interpretation limits ---")
print("1. In these tested configurations, ranking metrics are similar; this comparison does not establish a universal effect of weighting or SMOTE.")
print("2. Costs depend on the illustrative assumptions and selected thresholds; results should not be generalized beyond this dataset and setup.")

# Model Interpretation: SHAP Attributions and Illustrative Risk Drivers
**Illustrative model-explanation scope (not a compliance assessment):**
- **TreeExplainer Integration:** We compute SHAP feature attributions using a stratified sample of $5,000$ holdout test applicants ($N = 5,000$) to evaluate both global risk drivers and local decision explanations without out-of-memory bottlenecks.
- **Explanation outputs:**
  1. Mean $\vert{}SHAP\vert{}$ global feature importance bar plot (`assets/shap_bar.png`).
  2. Global beeswarm distribution plot showing risk directionality (`assets/shap_beeswarm.png`).
  3. Feature dependence plots for `EXT_SOURCE_2` and debt leverage (`assets/shap_dependence.png`).
  4. Illustrative local risk-driver summaries for 3 representative applicants: High-Risk ($p \ge 0.60$), Borderline Marginal Decline ($p \approx t^*$), and Low-Risk Approved ($p \le 0.05$).
  5. Descriptive comparison: Spearman rank correlation between mean $\vert{}SHAP\vert{}$ and standardized Logistic Regression coefficients to confirm linear/non-linear agreement.

In [ ]:
# Cell 18: Stratified Sample Preparation & TreeExplainer Execution
import shap
from scipy.stats import spearmanr

print("Preparing SHAP explanations for the selected tree model...")

# 1. Stratified 5,000-row sample from holdout test set to maintain representative prevalence
X_test_sample, _, y_test_sample, _ = train_test_split(
    X_test,
    y_test,
    train_size=5000,
    random_state=RANDOM_SEED,
    stratify=y_test,
)

# 2. Use the exact fitted preprocessor from the selected champion pipeline.
champ_preprocessor = champ_pipeline.named_steps["preprocessor"]
tree_feature_names = list(champ_preprocessor.get_feature_names_out())
numeric_cols = X_test.select_dtypes(include=[np.number]).columns.tolist()
X_test_sample_proc = champ_preprocessor.transform(X_test_sample)
X_test_sample_df = pd.DataFrame(X_test_sample_proc, columns=tree_feature_names, index=X_test_sample.index)

# 3. Retrieve underlying fitted Champion tree estimator
champion_model = champ_pipeline.named_steps["classifier"]

# 4. Compute TreeExplainer Shapley values
explainer = shap.TreeExplainer(champion_model)
shap_explanation = explainer(X_test_sample_df)

# Handle dimension output across SHAP versions
if len(shap_explanation.values.shape) == 3:
    shap_values = shap_explanation.values[:, :, 1]  # Default class attribution
else:
    shap_values = shap_explanation.values

print(f"SHAP explanation matrix computed: {shap_values.shape[0]:,} samples x {shap_values.shape[1]} features.")

## 1. Global Feature Attribution: Bar Plot, Beeswarm & Dependence Plots
**Model Risk Audit Findings:**  
We generate and archive the three statutory explainability figures into `assets/`:
1. `assets/shap_bar.png`: Global ranking of top risk-discriminating attributes based on mean absolute SHAP value.
2. `assets/shap_beeswarm.png`: Feature impact directionality (verifying that lower external bureau ratings and higher leverage push default log-odds upward).
3. `assets/shap_dependence.png`: Non-linear interaction profiles for `EXT_SOURCE_2` and `LOAN_TO_INCOME_RATIO`.

In [ ]:
# Cell 19: SHAP Visualizations Export (Bar, Beeswarm, Dependence)
plt.style.use("default")
plt.rcParams["font.sans-serif"] = "DejaVu Sans"

# Figure 1: Mean |SHAP| Bar Plot
fig_bar = plt.figure(figsize=(9, 6), dpi=120)
shap.summary_plot(shap_values, X_test_sample_df, plot_type="bar", show=False, max_display=15)
plt.title("Top 15 Credit Risk Drivers (Mean |SHAP|)", fontsize=12, fontweight="bold", pad=12)
plt.xlabel("Mean |SHAP Value| (Impact on Default Log-Odds)", fontsize=10)
plt.tight_layout()
bar_plot_path = ASSETS_DIR / "shap_bar.png"
plt.savefig(bar_plot_path, dpi=300)
plt.show()
print("Feature importance bar plot saved to assets/shap_bar.png")

# Figure 2: Global Beeswarm Plot
fig_bee = plt.figure(figsize=(9, 6), dpi=120)
shap.summary_plot(shap_values, X_test_sample_df, show=False, max_display=15)
plt.title("SHAP Global Summary: Feature Risk Impact Directionality", fontsize=12, fontweight="bold", pad=12)
plt.tight_layout()
beeswarm_plot_path = ASSETS_DIR / "shap_beeswarm.png"
plt.savefig(beeswarm_plot_path, dpi=300)
plt.show()
print("Global beeswarm plot saved to assets/shap_beeswarm.png")

# Figure 3: Dual Dependence Plots (EXT_SOURCE_2 & LOAN_TO_INCOME_RATIO)
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)
feature_col_1 = "EXT_SOURCE_2"
feature_col_2 = "LOAN_TO_INCOME_RATIO"

col_idx_1 = tree_feature_names.index(feature_col_1)
col_idx_2 = tree_feature_names.index(feature_col_2)

# Subplot A: EXT_SOURCE_2
shap.dependence_plot(
    col_idx_1, shap_values, X_test_sample_df.values,
    feature_names=tree_feature_names, ax=axes[0], show=False
)
axes[0].set_title(f"SHAP Dependence: {feature_col_1}", fontsize=11, fontweight="bold")

# Subplot B: LOAN_TO_INCOME_RATIO
shap.dependence_plot(
    col_idx_2, shap_values, X_test_sample_df.values,
    feature_names=tree_feature_names, ax=axes[1], show=False
)
axes[1].set_title(f"SHAP Dependence: {feature_col_2}", fontsize=11, fontweight="bold")

plt.tight_layout()
dependence_plot_path = ASSETS_DIR / "shap_dependence.png"
plt.savefig(dependence_plot_path, dpi=300)
plt.show()
print("SHAP dependence plots saved to assets/shap_dependence.png")

## 2. Local Attribution: Illustrative Risk Drivers
**Important:** This is an educational model-explanation exercise, not legal or regulatory advice.  
We select three illustrative holdout cases and list their top three positive SHAP attributions for the model's raw output. These are not consumer notices, legal adverse-action reasons, or causal explanations:
1. **High-risk:** sample with highest estimated risk.
2. **Near threshold:** sample closest to the selected threshold.
3. **Low-risk:** sample with lowest estimated risk.

In [ ]:
# Cell 20: Illustrative local risk-driver summaries
# Illustrative plain-language descriptions of risk drivers; not legal adverse-action reasons
RISK_DRIVER_MAP = {
    "EXT_SOURCE_1": "External risk score 1 contributes to the model's estimated default risk",
    "EXT_SOURCE_2": "External risk score 2 contributes to the model's estimated default risk",
    "EXT_SOURCE_3": "External risk score 3 contributes to the model's estimated default risk",
    "EXT_SOURCE_MEAN": "Composite external risk-score feature contributes to estimated default risk",
    "LOAN_TO_INCOME_RATIO": "Recorded loan amount is high relative to reported annual income",
    "ANNUITY_TO_INCOME": "Monthly loan payment obligation represents an excessive share of income",
    "CREDIT_TERM": "Loan payment-to-credit term structure indicates elevated amortization burden",
    "CREDIT_TO_GOODS": "Loan amount exceeds the appraised market price of financed goods",
    "YEARS_EMPLOYED": "Limited verified employment tenure in current occupation",
    "EMPLOYED_ANOM": "Absence of active formal employment or unverified labor status",
    "NAME_EDUCATION_TYPE": "Completed education level below prime underwriting preferred tier",
    "OCCUPATION_TYPE": "Occupational category associated with cyclical earnings volatility",
    "AMT_CREDIT": "Total requested principal amount exceeds credit committee portfolio guidelines",
    "AMT_ANNUITY": "Recorded monthly payment amount contributes to estimated default risk",
}

# 1. Get calibrated test probabilities for the 5,000 sampled rows
sample_indices = X_test_sample.index
sample_calib_probs = test_calib_probs[X_test.index.get_indexer(sample_indices)]

# 2. Select 3 representative applicants
# Applicant A: High-Risk (Highest probability in sample)
high_risk_loc = int(np.argmax(sample_calib_probs))

# Applicant B: Borderline (Probability closest to empirical t* = 0.1600)
borderline_loc = int(np.argmin(np.abs(sample_calib_probs - emp_t_star)))

# Applicant C: Low-Risk (Lowest probability in sample)
low_risk_loc = int(np.argmin(sample_calib_probs))

selected_cases = [
    ("High-Risk", high_risk_loc),
    ("Borderline Near Threshold", borderline_loc),
    ("Low-Risk", low_risk_loc),
]

reason_records = []
for label, loc_idx in selected_cases:
    applicant_prob = sample_calib_probs[loc_idx]
    applicant_shaps = shap_values[loc_idx]

    # Rank features by positive SHAP (driving default log-odds UP)
    risk_increasing_idx = np.argsort(applicant_shaps)[::-1]

    reasons = []
    for rank, feat_idx in enumerate(risk_increasing_idx[:3], 1):
        feat_name = tree_feature_names[feat_idx]
        feat_val = X_test_sample_df.iloc[loc_idx, feat_idx]
        shap_val = applicant_shaps[feat_idx]
        risk_driver_text = RISK_DRIVER_MAP.get(feat_name, f"Adverse risk contribution from attribute: {feat_name}")

        reason_records.append({
            "applicant_cohort": label,
            "calibrated_prob": round(float(applicant_prob), 4),
            "underwriting_action": "Decline" if applicant_prob >= emp_t_star else "Approve",
            "risk_factor_rank": rank,
            "feature_name": feat_name,
            "feature_value": round(float(feat_val), 4) if pd.notnull(feat_val) else "Missing",
            "shap_attribution": round(float(shap_val), 4),
            "illustrative_risk_driver": risk_driver_text,
        })

df_reasons = pd.DataFrame(reason_records)
reasons_csv_path = RESULTS_DIR / "shap_reason_codes.csv"
df_reasons.to_csv(reasons_csv_path, index=False)

print("\n=========================================================================================")
print("                   ILLUSTRATIVE RISK DRIVER REPORT                                      ")
print("=========================================================================================")
display_cols = ["applicant_cohort", "calibrated_prob", "underwriting_action", "risk_factor_rank", "feature_name", "shap_attribution", "illustrative_risk_driver"]
print(df_reasons[display_cols].to_string(index=False))
print("=========================================================================================")
print(f"Illustrative risk-driver table saved: {reasons_csv_path}")

## 3. Descriptive Spearman Rank Comparison ($\rho$)
**Descriptive comparison (linear versus tree model):**  
To address internal audit concerns regarding black-box complexity, we compare feature rankings between:
1. Mean absolute SHAP values ($\overline{|\text{SHAP}|}$) from the selected tree-model champion.
2. Absolute standardized coefficients ($|\beta_j|$) from the full regularized Logistic Regression.

The correlation describes rank concordance only. It does not prove shared causal or structural signals; collinearity and preprocessing differences limit interpretation.

In [ ]:
# Cell 21: Descriptive comparison of model feature rankings
from scipy.stats import spearmanr

# 1. Fit full Logistic Regression pipeline to extract standardized coefficients
lr_full_pipe = make_lr_full_pipe()
lr_full_pipe.fit(X_train, y_train)

lr_classifier = lr_full_pipe.named_steps["classifier"]
lr_coefs = lr_classifier.coef_[0]

# 2. Extract linear model feature names post-transformation
lr_transformer = lr_full_pipe.named_steps["preprocessor"]
lr_feature_names = list(lr_transformer.get_feature_names_out())

# 3. Match feature importances using EXACT feature names
comparison_records = []
mean_abs_shaps = np.mean(np.abs(shap_values), axis=0)

for feat in numeric_cols:
    if feat in tree_feature_names:
        tree_idx = tree_feature_names.index(feat)
        shap_imp = mean_abs_shaps[tree_idx]

        # Exact match (accounting for optional prefix like 'num__')
        matched_idx = [
            i for i, name in enumerate(lr_feature_names)
            if name == feat or name == f"num__{feat}"
        ]

        if matched_idx:
            lr_weight = np.abs(lr_coefs[matched_idx[0]])
            comparison_records.append({
                "feature": feat,
                "mean_abs_shap": float(shap_imp),
                "abs_lr_coef": float(lr_weight),
            })

df_rank_check = pd.DataFrame(comparison_records)

# 4. Compute Spearman Rank Correlation
rho, p_value = spearmanr(df_rank_check["mean_abs_shap"], df_rank_check["abs_lr_coef"])

# Add individual model ranks for auditing
df_rank_check["shap_rank"] = df_rank_check["mean_abs_shap"].rank(ascending=False).astype(int)
df_rank_check["lr_rank"] = df_rank_check["abs_lr_coef"].rank(ascending=False).astype(int)
df_rank_check = df_rank_check.sort_values(by="shap_rank").reset_index(drop=True)

# 5. Save artifacts to results/ as mandated by current project
rank_csv_path = RESULTS_DIR / "shap_lr_rank_correlation.csv"
df_rank_check.to_csv(rank_csv_path, index=False)

# Robustness check: keep one representative from each *_AVG/*_MODE/*_MEDI triplet.
import re
feature_groups = {}
for feature in df_rank_check["feature"].tolist():
    match = re.match(r"^(.*)_(AVG|MODE|MEDI)$", feature)
    if match:
        feature_groups.setdefault(match.group(1), []).append(feature)
representative_features = {
    feature
    for feature in df_rank_check["feature"].tolist()
    if not re.match(r"^(.*)_(AVG|MODE|MEDI)$", feature)
}
for group_features in feature_groups.values():
    representative_features.add(
        sorted(group_features, key=lambda value: (
            0 if value.endswith("_AVG") else
            1 if value.endswith("_MODE") else 2,
            value,
        ))[0]
    )
df_rank_robust = df_rank_check[df_rank_check["feature"].isin(representative_features)].copy()
rho_robust, p_value_robust = spearmanr(
    df_rank_robust["mean_abs_shap"], df_rank_robust["abs_lr_coef"]
)
df_rank_robust["shap_rank"] = df_rank_robust["mean_abs_shap"].rank(ascending=False).astype(int)
df_rank_robust["lr_rank"] = df_rank_robust["abs_lr_coef"].rank(ascending=False).astype(int)
df_rank_robust = df_rank_robust.sort_values(by="shap_rank").reset_index(drop=True)
df_rank_robust.to_csv(RESULTS_DIR / "shap_lr_rank_correlation_robust.csv", index=False)

rank_summary_txt_path = RESULTS_DIR / "rank_correlation.txt"
with open(rank_summary_txt_path, "w", encoding="utf-8") as f:
    f.write(f"Spearman rank correlation (all shared numeric features): {rho:.4f}\n")
    f.write(f"p-value (all shared numeric features): {p_value:.4e}\n")
    f.write(f"Evaluated shared numeric features: {len(df_rank_check)}\n")
    f.write(f"Spearman rank correlation (one representative per AVG/MODE/MEDI triplet): {rho_robust:.4f}\n")
    f.write(f"p-value (triplet-reduced set): {p_value_robust:.4e}\n")
    f.write(f"Evaluated features after triplet reduction: {len(df_rank_robust)}\n")

print("=========================================================================================")
print("            DESCRIPTIVE RANK COMPARISON: TREE CHAMPION VS. LOGISTIC REGRESSION               ")
print("=========================================================================================")
print(f"• Shared numeric features compared : {len(df_rank_check)} continuous risk factors")
print(f"• Spearman Rank Correlation (all features) : {rho:.4f} (p={p_value:.4e})")
print(f"• Spearman Rank Correlation (triplet-reduced): {rho_robust:.4f} (p={p_value_robust:.4e})")
print(f"• Features after triplet reduction          : {len(df_rank_robust)}")
print("=========================================================================================")
print(f"Rank-correlation number and table archived to:\n• {rank_summary_txt_path}\n• {rank_csv_path}")

print("\n--- Top 10 Risk Factors: Champion (SHAP) vs. Baseline (Standardized LR Coef) ---")
display_cols = ["shap_rank", "feature", "mean_abs_shap", "lr_rank", "abs_lr_coef"]
print(df_rank_check[display_cols].head(10).to_string(index=False))

# Descriptive comparison only; no statistical threshold establishes that models agree
print("\nRank agreement is descriptive only; significance does not establish feature equivalence or model validity.")

# Bureau-History Feature Engineering
**Feature source and aggregation:**
- **Data Source:** `data/raw/bureau.csv` ($1,716,428$ records across all reported multi-lender credit history).
- **Engineered Bureau Attributes:**
  - Loan Volume: `BUREAU_LOAN_COUNT`, `BUREAU_ACTIVE_LOAN_COUNT`, `BUREAU_CLOSED_LOAN_COUNT`.
  - Recency & Maturity: `BUREAU_DAYS_CREDIT_MEAN`, `BUREAU_DAYS_CREDIT_MAX`.
  - Exposure & Leverage: `BUREAU_AMT_CREDIT_SUM`, `BUREAU_AMT_CREDIT_SUM_DEBT`, `BUREAU_DEBT_CREDIT_RATIO`.
  - Delinquency Burden: `BUREAU_CREDIT_DAY_OVERDUE_MAX`, `BUREAU_AMT_CREDIT_SUM_OVERDUE`.
  - Portfolio History Flag: `HAS_BUREAU` ($1$ for matched applicants, $0$ for unbanked thin-files).
- **Joining:** Left-join to `X_train` and `X_test` via `SK_ID_CURR`. Applicants with no bureau history have count features imputed to $0$.

In [ ]:
# Cell 22: P6 Bureau Feature Ingestion & Relational Merging
from src.bureau_features import build_bureau_features

print("Extracting credit bureau feature store...")
df_bureau_features = build_bureau_features()

# Retrieve matching SK_ID_CURR from raw dataframe
train_ids = df_raw.loc[X_train.index, "SK_ID_CURR"]
test_ids = df_raw.loc[X_test.index, "SK_ID_CURR"]

# Merge onto Training Set
df_train_id = pd.DataFrame({"SK_ID_CURR": train_ids}, index=X_train.index)
train_bureau_merged = df_train_id.merge(df_bureau_features, on="SK_ID_CURR", how="left").set_index(X_train.index)
train_bureau_merged = train_bureau_merged.drop(columns=["SK_ID_CURR"])

# Merge onto Test Set
df_test_id = pd.DataFrame({"SK_ID_CURR": test_ids}, index=X_test.index)
test_bureau_merged = df_test_id.merge(df_bureau_features, on="SK_ID_CURR", how="left").set_index(X_test.index)
test_bureau_merged = test_bureau_merged.drop(columns=["SK_ID_CURR"])

# Zero-fill count and binary history flags for unbanked applicants
count_cols = ["BUREAU_LOAN_COUNT", "BUREAU_ACTIVE_LOAN_COUNT", "BUREAU_CLOSED_LOAN_COUNT", "HAS_BUREAU"]
for c in count_cols:
    train_bureau_merged[c] = train_bureau_merged[c].fillna(0)
    test_bureau_merged[c] = test_bureau_merged[c].fillna(0)

# Construct Combined Feature Matrices (+Bureau)
X_train_bureau = pd.concat([X_train, train_bureau_merged], axis=1)
X_test_bureau = pd.concat([X_test, test_bureau_merged], axis=1)

print("\n--- Feature Matrix Dimensionality Audit ---")
print(f"• Application-Only Matrix : {X_train.shape[1]} features")
print(f"• Application + Bureau    : {X_train_bureau.shape[1]} features (+{train_bureau_merged.shape[1]} bureau indicators)")
print(f"• Bureau History Coverage : {train_bureau_merged['HAS_BUREAU'].mean():.2%} of applicants have credit bureau records")

In [ ]:
# Holdout segmentation by availability of a linked bureau record.
# Descriptive evaluation only; this is not a fairness or causal analysis.
from src.evaluate import calculate_risk_metrics
segment_flag = test_bureau_merged["HAS_BUREAU"].fillna(0).to_numpy(dtype=int)
y_test_array = y_test.to_numpy(dtype=int)
thin_file_rows = []
for flag, segment_name in [(0, "No linked bureau record"), (1, "Linked bureau record")]:
    mask = segment_flag == flag
    y_seg = y_test_array[mask]
    p_seg = test_calib_probs[mask]
    credit_seg = X_test.loc[test_bureau_merged.index[mask], "AMT_CREDIT"].to_numpy()
    predicted_seg = (p_seg >= emp_t_star).astype(int)
    operating = compute_credit_cost(
        y_seg, p_seg, threshold=emp_t_star, amt_credit=credit_seg
    )
    metric = calculate_risk_metrics(y_seg, p_seg) if np.unique(y_seg).size == 2 else {}
    thin_file_rows.append({
        "segment": segment_name,
        "n_applicants": int(mask.sum()),
        "default_prevalence": float(y_seg.mean()) if len(y_seg) else np.nan,
        "roc_auc": metric.get("roc_auc", np.nan),
        "pr_auc": metric.get("pr_auc", np.nan),
        "brier_score": metric.get("brier_score", np.nan),
        "approval_rate": operating["approval_rate"],
        "default_rate_approved": operating["default_rate_approved"],
        "recall": operating["recall"],
        "precision": operating["precision"],
        "expected_decision_cost_per_applicant": operating["cost_per_applicant"],
    })
thin_file_segments = pd.DataFrame(thin_file_rows)
thin_file_segments.to_csv(RESULTS_DIR / "thin_file_segments.csv", index=False)
print("Holdout performance by bureau-record coverage saved to results/thin_file_segments.csv")
print(thin_file_segments.to_string(index=False))

## 1. Bureau-Feature Ablation
**Ablation Hypothesis:**  
We compare OOF PR-AUC and ROC-AUC for the selected champion with application-only features versus application plus bureau aggregates. Fold-level changes are saved for inspection; this is not a significance test.

In [ ]:
# Cell 23: Cross-validation ablation for the selected champion with bureau features
from src.features import get_preprocessors
from sklearn.base import clone

_, tree_preprocessor_bureau = get_preprocessors(X_train_bureau)
def make_champion_bureau_pipe():
    if champion_name == "xgboost":
        estimator = xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05, min_child_weight=10,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=scale_pos_weight,
            tree_method="hist", random_state=RANDOM_SEED, n_jobs=-1, eval_metric="logloss",
        )
    else:
        estimator = lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=100,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
        )
    return Pipeline([("preprocessor", clone(tree_preprocessor_bureau)), ("classifier", estimator)])

print(f"Running 5-fold CV for selected champion ({champion_name}) with bureau features...")
oof_champion_bureau, scores_champion_bureau, folds_champion_bureau = cv_oof_evaluate(
    model_name=f"{champion_name}_bureau",
    pipeline_factory=make_champion_bureau_pipe,
    X=X_train_bureau,
    y=y_train,
    save_oof=True,
)
score_app_only = df_cv_results[df_cv_results["model"] == champion_name].iloc[0]
pr_auc_bureau_lift = ((scores_champion_bureau["pr_auc"] - score_app_only["pr_auc"]) / score_app_only["pr_auc"]) * 100
roc_auc_bureau_lift = ((scores_champion_bureau["roc_auc"] - score_app_only["roc_auc"]) / score_app_only["roc_auc"]) * 100
print(f"Application-only PR-AUC: {score_app_only['pr_auc']:.4f}; with bureau: {scores_champion_bureau['pr_auc']:.4f}")
print(f"Application-only ROC-AUC: {score_app_only['roc_auc']:.4f}; with bureau: {scores_champion_bureau['roc_auc']:.4f}")
print(f"Relative PR-AUC lift: {pr_auc_bureau_lift:+.2f}%")
print("Lift is descriptive until per-fold results are examined.")

# Previous-Application History Features
**Previous-application history (`previous_application.csv`):**
- **Data Source:** $1,670,214$ historical internal loan origination decisions.
- **Data Handling:** Approved-application amount features exclude refused/cancelled applications. Missing amounts are not automatically interpreted as zero.
- **Engineered Relationship Features:**
  - Origination Volume: `PREV_APP_COUNT`, `PREV_APPROVED_COUNT`, `PREV_REFUSED_COUNT`.
  - Relationship Health: `PREV_REFUSAL_RATE` (Internal declination ratio).
  - Financing Aggressiveness: `PREV_AMT_APPLICATION_MEAN`, `PREV_AMT_CREDIT_MEAN`, `PREV_CREDIT_APP_RATIO_MEAN`.
  - Recency: `PREV_DAYS_DECISION_MAX` (Days elapsed since most recent internal credit interaction).
  - Internal History Flag: `HAS_PREV`.

In [ ]:
# Cell 24: P7 Previous Application Ingestion & Relational Merging
from src.bureau_features import build_previous_app_features

print("Extracting previous application feature store...")
df_prev_features = build_previous_app_features()

# Merge onto Training and Test sets
train_prev_merged = df_train_id.merge(df_prev_features, on="SK_ID_CURR", how="left").set_index(X_train.index)
train_prev_merged = train_prev_merged.drop(columns=["SK_ID_CURR"])

test_prev_merged = df_test_id.merge(df_prev_features, on="SK_ID_CURR", how="left").set_index(X_test.index)
test_prev_merged = test_prev_merged.drop(columns=["SK_ID_CURR"])

# Zero-fill count features for first-time applicants
prev_count_cols = ["PREV_APP_COUNT", "PREV_APPROVED_COUNT", "PREV_REFUSED_COUNT", "HAS_PREV"]
for c in prev_count_cols:
    train_prev_merged[c] = train_prev_merged[c].fillna(0)
    test_prev_merged[c] = test_prev_merged[c].fillna(0)

# Construct Feature Matrices
# 1. Application + Previous Only
X_train_prev = pd.concat([X_train, train_prev_merged], axis=1)
X_test_prev = pd.concat([X_test, test_prev_merged], axis=1)

# 2. Application + Both (Bureau + Previous)
X_train_both = pd.concat([X_train_bureau, train_prev_merged], axis=1)
X_test_both = pd.concat([X_test_bureau, test_prev_merged], axis=1)

print("\n--- Relational Feature Store Summary ---")
print(f"• Application-Only Matrix           : {X_train.shape[1]} columns")
print(f"• Application + Bureau Matrix       : {X_train_bureau.shape[1]} columns")
print(f"• Application + Previous Matrix     : {X_train_prev.shape[1]} columns")
print(f"• Full Relational Matrix (+ Both)   : {X_train_both.shape[1]} columns")
print(f"• Internal History Coverage         : {train_prev_merged['HAS_PREV'].mean():.2%} of applicants have prior loans with us")

## 2. Four-Way Relational Feature Ablation
**Evaluation:**  
We execute 5-fold cross-validation on the selected champion across four feature sets:
1. **Application-Only:** Baseline thin-file scorecard.
2. **+ Bureau Only:** Application + External Credit Bureau Tradelines.
3. **+ Previous Only:** Application + Internal Bank Origination History.
4. **+ Both (Comprehensive Store):** Combined application, bureau, and previous-application features.

In [ ]:
# Cell 25: Cross-validation for previous-only and full relational features
_, tree_prep_prev = get_preprocessors(X_train_prev)
def make_champion_prev_pipe():
    if champion_name == "xgboost":
        estimator = xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05, min_child_weight=10,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=scale_pos_weight,
            tree_method="hist", random_state=RANDOM_SEED, n_jobs=-1, eval_metric="logloss",
        )
    else:
        estimator = lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=100,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
        )
    return Pipeline([("preprocessor", clone(tree_prep_prev)), ("classifier", estimator)])

oof_champion_prev, scores_champion_prev, folds_champion_prev = cv_oof_evaluate(
    model_name=f"{champion_name}_prev",
    pipeline_factory=make_champion_prev_pipe,
    X=X_train_prev,
    y=y_train,
    save_oof=True,
)

_, tree_prep_both = get_preprocessors(X_train_both)
def make_champion_both_pipe():
    if champion_name == "xgboost":
        estimator = xgb.XGBClassifier(
            n_estimators=500, max_depth=5, learning_rate=0.05, min_child_weight=10,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=scale_pos_weight,
            tree_method="hist", random_state=RANDOM_SEED, n_jobs=-1, eval_metric="logloss",
        )
    else:
        estimator = lgb.LGBMClassifier(
            n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=100,
            subsample=0.8, colsample_bytree=0.7, scale_pos_weight=scale_pos_weight,
            random_state=RANDOM_SEED, n_jobs=-1, verbose=-1,
        )
    return Pipeline([("preprocessor", clone(tree_prep_both)), ("classifier", estimator)])

oof_champion_both, scores_champion_both, folds_champion_both = cv_oof_evaluate(
    model_name=f"{champion_name}_both",
    pipeline_factory=make_champion_both_pipe,
    X=X_train_both,
    y=y_train,
    save_oof=True,
)
print("Previous-only and full-store cross-validation completed for selected champion:", champion_name)

## 3. Official Relational Ablation Scorecard & Decision Gate
**Output:**  
We construct the four-row comparative matrix, save `results/ablation_study.csv`, and evaluate whether multi-table features improve fold-level scores over the thin-file application model.

In [ ]:
# Cell 26: Relational ablation tables and paired fold differences
ablation_records = [
    {
        "feature_set": "Application-only",
        "num_features": X_train.shape[1],
        "pr_auc": score_app_only["pr_auc"],
        "pr_auc_std": score_app_only["pr_auc_std"],
        "roc_auc": score_app_only["roc_auc"],
        "roc_auc_std": score_app_only["roc_auc_std"],
        "gini": score_app_only["gini"],
        "ks_stat": score_app_only["ks_stat"],
        "relative_pr_lift_pct": 0.0,
    },
    {
        "feature_set": "+ Bureau",
        "num_features": X_train_bureau.shape[1],
        "pr_auc": scores_champion_bureau["pr_auc"],
        "pr_auc_std": scores_champion_bureau["pr_auc_std"],
        "roc_auc": scores_champion_bureau["roc_auc"],
        "roc_auc_std": scores_champion_bureau["roc_auc_std"],
        "gini": scores_champion_bureau["gini"],
        "ks_stat": scores_champion_bureau["ks_stat"],
        "relative_pr_lift_pct": pr_auc_bureau_lift,
    },
    {
        "feature_set": "+ Previous",
        "num_features": X_train_prev.shape[1],
        "pr_auc": scores_champion_prev["pr_auc"],
        "pr_auc_std": scores_champion_prev["pr_auc_std"],
        "roc_auc": scores_champion_prev["roc_auc"],
        "roc_auc_std": scores_champion_prev["roc_auc_std"],
        "gini": scores_champion_prev["gini"],
        "ks_stat": scores_champion_prev["ks_stat"],
        "relative_pr_lift_pct": ((scores_champion_prev["pr_auc"] - score_app_only["pr_auc"]) / score_app_only["pr_auc"]) * 100,
    },
    {
        "feature_set": "+ Both (Full Store)",
        "num_features": X_train_both.shape[1],
        "pr_auc": scores_champion_both["pr_auc"],
        "pr_auc_std": scores_champion_both["pr_auc_std"],
        "roc_auc": scores_champion_both["roc_auc"],
        "roc_auc_std": scores_champion_both["roc_auc_std"],
        "gini": scores_champion_both["gini"],
        "ks_stat": scores_champion_both["ks_stat"],
        "relative_pr_lift_pct": ((scores_champion_both["pr_auc"] - score_app_only["pr_auc"]) / score_app_only["pr_auc"]) * 100,
    },
]

df_ablation = pd.DataFrame(ablation_records)
ablation_csv_path = RESULTS_DIR / "ablation_study.csv"
df_ablation.to_csv(ablation_csv_path, index=False)

# Compare the same validation folds for each feature-set pair.
folds_app = folds_xgb if champion_name == "xgboost" else folds_lgb
paired_fold_records = []
for label, added_folds in [
    ("+ Bureau", folds_champion_bureau),
    ("+ Previous", folds_champion_prev),
    ("+ Both", folds_champion_both),
]:
    pair = pd.DataFrame({
        "fold": folds_app["fold"].to_numpy(),
        "feature_set": label,
        "pr_auc_application_only": folds_app["pr_auc"].to_numpy(),
        "pr_auc_augmented": added_folds["pr_auc"].to_numpy(),
        "pr_auc_delta": added_folds["pr_auc"].to_numpy() - folds_app["pr_auc"].to_numpy(),
        "roc_auc_application_only": folds_app["roc_auc"].to_numpy(),
        "roc_auc_augmented": added_folds["roc_auc"].to_numpy(),
        "roc_auc_delta": added_folds["roc_auc"].to_numpy() - folds_app["roc_auc"].to_numpy(),
    })
    paired_fold_records.append(pair)
df_ablation_fold_diffs = pd.concat(paired_fold_records, ignore_index=True)
df_ablation_fold_diffs.to_csv(RESULTS_DIR / "ablation_fold_diffs.csv", index=False)
print("Fold-level paired differences saved to results/ablation_fold_diffs.csv")

# Fold-aligned comparison of the two gradient-boosted models.
champion_folds = folds_xgb if champion_name == "xgboost" else folds_lgb
runner_up_folds = folds_xgb if runner_up_name == "xgboost" else folds_lgb
df_champion_runnerup_fold_diffs = pd.DataFrame({
    "fold": champion_folds["fold"].to_numpy(),
    "champion": champion_name,
    "runner_up": runner_up_name,
    "champion_pr_auc": champion_folds["pr_auc"].to_numpy(),
    "runner_up_pr_auc": runner_up_folds["pr_auc"].to_numpy(),
    "pr_auc_delta_champion_minus_runner_up": champion_folds["pr_auc"].to_numpy() - runner_up_folds["pr_auc"].to_numpy(),
    "champion_roc_auc": champion_folds["roc_auc"].to_numpy(),
    "runner_up_roc_auc": runner_up_folds["roc_auc"].to_numpy(),
    "roc_auc_delta_champion_minus_runner_up": champion_folds["roc_auc"].to_numpy() - runner_up_folds["roc_auc"].to_numpy(),
})
df_champion_runnerup_fold_diffs.to_csv(RESULTS_DIR / "champion_runnerup_fold_diffs.csv", index=False)
print("Champion-versus-runner-up fold differences saved to results/champion_runnerup_fold_diffs.csv")
print(df_champion_runnerup_fold_diffs.to_string(index=False))

print("\n=========================================================================================")
print("                   RELATIONAL FEATURE ABLATION SCORECARD                    ")
print("=========================================================================================")
print(df_ablation[["feature_set", "num_features", "pr_auc", "roc_auc", "gini", "ks_stat", "relative_pr_lift_pct"]].to_string(index=False))
print("=========================================================================================")
print(f"Ablation scorecard successfully archived to: {ablation_csv_path}")

In [ ]:
# Feature-policy ablation: compare with the four additional sensitive/proxy fields retained.
# CODE_GENDER remains excluded in both variants. This is a predictive sensitivity analysis,
# not a legal fair-lending test or a finding of compliance.
X_policy_original, _ = build_features(df_raw, drop_fair_lending=False)
X_policy_train = X_policy_original.loc[X_train.index]
policy_lr_preprocessor, policy_tree_preprocessor = get_preprocessors(X_policy_train)
from sklearn.base import clone
from sklearn.pipeline import Pipeline
from src.evaluate import cv_oof_evaluate
def make_policy_ablation_pipeline():
    return Pipeline([
        ("preprocessor", clone(policy_tree_preprocessor)),
        ("classifier", clone(champ_pipeline.named_steps["classifier"])),
    ])
_, policy_scores, policy_fold_metrics = cv_oof_evaluate(
    model_name="champion_policy_ablation",
    pipeline_factory=make_policy_ablation_pipeline,
    X=X_policy_train,
    y=y_train,
    save_oof=False,
)
fair_lending_ablation = pd.DataFrame([
    {
        "policy": "Configured feature exclusions",
        "num_features": int(X_train.shape[1]),
        "pr_auc": float(champion_pr_auc),
        "roc_auc": float(champion_roc_auc),
    },
    {
        "policy": "Retain four additional sensitive/proxy features (CODE_GENDER still excluded)",
        "num_features": int(X_policy_train.shape[1]),
        "pr_auc": float(policy_scores["pr_auc"]),
        "roc_auc": float(policy_scores["roc_auc"]),
    },
])
fair_lending_ablation["pr_auc_delta_vs_configured"] = fair_lending_ablation["pr_auc"] - fair_lending_ablation.loc[0, "pr_auc"]
fair_lending_ablation["roc_auc_delta_vs_configured"] = fair_lending_ablation["roc_auc"] - fair_lending_ablation.loc[0, "roc_auc"]
fair_lending_ablation.to_csv(RESULTS_DIR / "fair_lending_ablation.csv", index=False)
print("Feature-policy sensitivity saved to results/fair_lending_ablation.csv")

In [ ]:
# Record configuration and artifact lineage for this execution.
import json
import platform
from importlib import metadata
package_versions = {}
for package in ["numpy", "pandas", "scikit-learn", "xgboost", "lightgbm", "shap", "imbalanced-learn", "scipy"]:
    try:
        package_versions[package] = metadata.version(package)
    except metadata.PackageNotFoundError:
        package_versions[package] = None
manifest = {
    "champion": champion_name,
    "runner_up": runner_up_name,
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "train_rows": int(len(X_train)),
    "test_rows": int(len(X_test)),
    "target_prevalence_train": float(y_train.mean()),
    "target_prevalence_test": float(y_test.mean()),
    "feature_counts": {
        "application_only": int(X_train.shape[1]),
        "application_plus_bureau": int(X_train_bureau.shape[1]),
        "application_plus_previous": int(X_train_prev.shape[1]),
        "full_relational_store": int(X_train_both.shape[1]),
    },
    "decision_cost_assumptions": {
        "lgd_reference_assumption": 0.45,
        "rejection_opportunity_cost_assumption": 0.09,
        "cost_ratio": 5.0,
        "currency": "dataset currency units; no currency symbol implied",
    },
    "fair_lending_policy": "Illustrative feature exclusions; no legal compliance or fairness conclusion",
    "hyperparameter_tuning": "Not performed; fixed configuration",
    "dataset": "Home Credit Default Risk; single dataset and split",
    "python_version": platform.python_version(),
    "package_versions": package_versions,
}
with open(RESULTS_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print("Run manifest saved to results/run_manifest.json")